# Etapas de un estudio de simulación

## ¿Por qué empezar con una definición  del sistema y el problema a resolver?

Cuando queremos usar la simulación para analizar o diseñar un sistema, la **tentación común** es abrir el computador y comenzar a programar. Sin embargo, sin una **definición clara** de lo que queremos simular, corremos un alto riesgo de:

* Modelar **el sistema equivocado**,
* Hacer suposiciones poco realistas,
* O terminar con resultados bonitos pero inútiles para la toma de decisiones.

Por eso, todo estudio de simulación debe comenzar con una **fase de definición y delimitación** del problema.

---

## El propósito de la simulación

Un **modelo de simulación** no es el sistema real, sino una **representación simplificada** que nos permite:

* **Comprender** cómo funciona el sistema,
* **Explorar escenarios** que serían costosos o imposibles en la realidad,
* **Apoyar decisiones** de diseño, planificación o mejora.

La simulación no busca predecir el futuro con certeza absoluta, sino dar información **confiable y útil** para tomar mejores decisiones.

---

## Pregunta inicial

Antes de iniciar, debemos responder:

**¿Qué decisión quiero apoyar con este estudio de simulación?**

Ejemplos:

* Decidir **cuántos servidores** necesita un sistema web para cumplir un SLA.
* Evaluar si una política de **autoscaling** reduce la latencia.
* Estimar la **capacidad de atención** de un hospital en emergencias.
* Predecir la **demanda energética** bajo distintos escenarios de crecimiento poblacional.

---

## Riesgos

Si omitimos etapas:

* Podemos **definir mal la frontera del sistema**, incluyendo o excluyendo variables críticas.
* Podríamos usar **hipótesis erróneas** (ejemplo: suponer tiempos de servicio exponenciales cuando en realidad son logaritmicos).
* Invertiríamos horas en codificar un modelo que **no responde la pregunta clave**.

**Ejemplo:**
Un equipo simuló el **backlog de tickets de soporte** suponiendo que la tasa de llegada era constante, pero en la práctica variaba con la hora del día. El modelo subestimó los picos y llevó a **subdimensionar personal**.

---

## Etapas iniciales de un estudio de simulación

La introducción es el **punto de partida** del ciclo de simulación. Sirve para:

1. **Definir el sistema**: qué está dentro y qué queda fuera.
2. **Plantear hipótesis iniciales**: qué supuestos simplifican la realidad.
3. **Identificar variables clave**: entradas, estado y salidas.
4. **Visualizar relaciones**: cómo se conectan los elementos del sistema.

De aquí en adelante, estas definiciones guiarán todo el trabajo: el diseño del modelo, la selección de técnicas y la interpretación de resultados.

---

## Preguntas claves

Piensa en un sistema cotidiano que uses (una aplicación móvil, un transporte público, un servicio en línea). Pregúntate:

* ¿Qué **decisión** tomaría yo si tuviera que mejorar este sistema?
* ¿Qué debo definir primero para que una simulación de ese sistema tenga sentido?

La simulación es una herramienta poderosa, pero su valor depende de cómo comencemos. Una introducción bien planteada asegura que el modelo responda a la pregunta correcta y que los resultados sean útiles.

---


# Definición del sistema

## Concepto clave

Un **sistema** es un conjunto de **elementos interrelacionados** que interactúan entre sí dentro de un **límite definido** (frontera) para cumplir un **propósito**. En simulación, definir un sistema significa **decidir qué aspectos de la realidad vamos a representar** y cuáles dejaremos fuera.

Si no delimitamos bien, corremos el riesgo de:

* Modelar fenómenos **irrelevantes** (complejidad innecesaria).
* Omitir factores **críticos** (modelo inservible).

---

## Componentes esenciales de un sistema

1. **Entradas**

   * Son los **flujos que llegan desde el exterior** y afectan el comportamiento del sistema.
   * Ejemplos: peticiones de usuarios, llegada de clientes, consumo de energía, recursos asignados.
   * En un modelo suelen representarse como **variables exógenas** ($\lambda$, tasa de llegada).

2. **Estado**

   * Es lo que el sistema **“recuerda”** en un momento dado.
   * Determina la evolución futura junto con las entradas.
   * Ejemplos: número de clientes en cola, temperatura actual de un procesador, backlog de tareas.
   * Matemáticamente: $x(t)$ (en sistemas dinámicos).

3. **Salidas**

   * Son las **métricas observables** o resultados de interés.
   * Ejemplos: latencia promedio, p95 de respuesta, throughput, costo mensual, tiempo de espera.
   * Determinan la **toma de decisiones**.

4. **Frontera**

   * Marca qué está **dentro** del modelo y qué se considera **parte del entorno externo**.
   * Ejemplo: en un sistema de mensajería instantánea, ¿incluyo el servidor de base de datos o solo el backend de mensajes?
   * Una frontera mal definida lleva a conclusiones engañosas.



---

## Esquema visual

<p align="center">
  <img src="Concep_system.jpg" alt="Componentes de un sistema" width="1000"/>
</p>

---

## Ejemplo

**Sistema:** API de un microservicio

* **Entradas:** peticiones de usuarios, asignación de CPU.
* **Estado:** número de solicitudes en cola, hilos ocupados.
* **Salidas:** latencia p95, tiempo promedio de respuesta, throughput.
* **Frontera:** se incluye solo el backend de microservicios; se excluyen la base de datos externa y el tráfico de otros sistemas.

---

## Ejercicio 1.

1. Elijan un sistema y delimitenlo en sus**Componentes esenciales**:

   * **Entradas**
   * **Estado**
   * **Salidas**
   * **Frontera**

---

## Guia pratica

Antes de simular, debemos **dibujar y describir el sistema con claridad**. Esta definición inicial guiará:

* Qué datos recolectar,
* Qué hipótesis formular,
* Qué técnicas de simulación aplicar.

Un **buen modelo** comienza con una **definición precisa del sistema**.



# Identificación de variables

## ¿Para qué identificar variables?

Antes de programar un modelo, necesitamos **saber qué medimos y qué controlamos**. La correcta identificación de variables:

* evita omisiones críticas,
* reduce complejidad innecesaria,
* y conecta el modelo con la **decisión** que queremos informar.

En términos generales, describimos un sistema con:

* **Entradas** $u$: lo que llega desde fuera o controlamos,
* **Estado** $x$: lo que el sistema “recuerda” y determina su evolución,
* **Salidas** $y$: indicadores observables para decidir.

> Notación útil (dinámico):
> $x_{t+1} = F(x_t, u_t, \theta, \varepsilon_t)$,
> $y_t = G(x_t, u_t, \theta, \eta_t)$.
> $\theta$: parámetros; $\varepsilon, \eta$: ruido/variabilidad.

---

## Clasificación de variables (con criterios prácticos)

### Variables de entrada (controlables o externas)

**Qué son:** factores **exógenos** o **palancas de control** que afectan el sistema.

* **Externas (no controlables):** demanda de usuarios $\lambda(t)$, clima, calendario.
* **Controlables (políticas):** número de servidores, reglas de prioridad, límites de tasa.

**Buenas prácticas:**

* Registrar **unidades** y **granularidad temporal** (por min/hora).
* Si varían con el tiempo, tratarlas como series $u(t)$ o por franjas (mañana/tarde/noche).

**Ejemplos:**

* Microservicio: $\lambda(t)$ llegadas/min; política de reintentos.
* Restaurante: tasa de llegada clientes/min; número de cajas abiertas.

---

### Variables de estado (evolución interna)

**Qué son:** memoria del sistema; junto a $u$ determinan el futuro.

* **Discretas:** longitud de cola $B(t)$. Por ejemplo, servidores ocupados o vehículos en fila.
* **Continuas:** temperatura CPU $T(t)$, nivel de tanque, posición/velocidad de un robot.

**Señales de buena elección del estado:**

* **Suficiencia:** con $x(t)$ y $u(t)$ puedes predecir el siguiente paso.
* **Coherencia física:** conservacion (entradas–salidas = variación del estado).
* **Cotas:** capacidades máximas (p. ej., cola limitada).

---

### Variables de salida

entre estas están las métricas para **tomar decisiones**.

* **Nivel de servicio:** latencia p95/p99, tiempo de espera, pérdida/abandono.
* **Eficiencia:** throughput, % de utilización, costo, consumo energético.
* **Estabilidad:** tiempo de estabilización, sobreimpulso (pico tras un cambio).

Además de promedios, reportar **percentiles** e **intervalos de confianza** (capturan variabilidad).

---

## Tipo de variables y temporalidad

* **Tipos:** booleana, categórica, ordinal, numérica (entera/real).
Aquí tienes **3 ejemplos para cada tipo de variable**:

   - Booleana (sí/no)
      * ¿El servidor está activo? (activo/inactivo)   
      * ¿La cola está vacía? (sí/no)
      * ¿Se superó el umbral de latencia? (sí/no)

   - Categórica (sin orden)
      * Tipo de solicitud: registro / consulta / pago
      * Medio de pago: efectivo / tarjeta / vale
      * Zona de atención: norte / centro / sur

   - Ordinal (con orden)
      * Prioridad del ticket: baja / media / alta
      * Nivel de servicio: básico / estándar / premium
      * Satisfacción del usuario: malo / regular / bueno / excelente

   - Numérica entera

      * Número de personas en cola
      * Número de servidores activos
      * Número de reintentos por solicitud

   - Numérica real
      * Latencia de respuesta (ms)
      * Temperatura del procesador (°C)
      * Tasa de llegadas (clientes por minuto)

* **Índice temporal:** por **eventos** o **pasos** (Δt).
* **Ventanas de cálculo:** usar ventanas de tiempo para métricas temporales (p. ej., p95 cada 5 min).
* **Calidad de datos:** tratar ausentes/outliers; documentar fuentes y unidades.

---

## Relaciones y restricciones que vinculan variables

* **Ley de Little** (sistemas abiertos, estado estable): $L = \lambda W$. donde L es el número promedio de elementos en el sistema (como clientes en una cola), λ es la tasa promedio de llegada de los elementos, y w es el tiempo promedio que los elementos pasan en el sistema. 
* **Sistemas cerrados:** $N = (población fija).
* **Balances**: entradas – salidas = cambio del estado.
* **Capacidad**: $0 \le B(t) \le B_{\max}$, utilización $0 \le \rho \le 1$.

---

## Ejemplo - restaurante universitario

**Contexto:** hora de almuerzo con llegadas variables.

* **Entrada:** tasa de llegada de estudiantes $\lambda(t)$ (no controlable); número de cajas abiertas $c(t)$ (controlable).
* **Estado:** número de personas en cola $B(t)$; servidores ocupados.
* **Salidas:** tiempo promedio de espera $W_q$, p95 de espera, throughput (clientes/min).

**Cómo medir y modelar:**

* Medir llegadas por minuto; estimar distribución de **tiempos de servicio** (¿exponencial? ¿lognormal?).
* Si hay **picos** marcados, usar $\lambda(t)$ por franja.
* Diseñar experimentos: abrir +1 caja desde 12:00 a 12:30 y observar impacto en p95.

---

## Otros ejemplos

**Microservicio web**

* Entrada: $\lambda(t)$ requests/min; política de autoscaling.
* Estado: longitud de cola, instancias activas.
* Salidas: p95 latencia, tasa de error, costo/hora.

**App de transporte**

* Entrada: demanda por zona/hora; oferta de conductores.
* Estado: vehículos libres/ocupados, colas virtuales.
* Salidas: tiempo de espera del usuario, tasa de cancelación.

**Biblioteca digital**

* Entrada: solicitudes de préstamo/acceso.
* Estado: recursos concurrentes en uso, cola de espera.
* Salidas: tiempo de atención, disponibilidad del servicio.

---

## Errores comunes y cómo evitarlos

* **Confundir entrada con salida:** p. ej., usar “latencia” como entrada cuando es resultado.
* **Estado insuficiente:** no incluir variables necesarias (p. ej., servidores ocupados).
* **Unidades inconsistentes:** mezcla de seg/min → resultados incoherentes.
* **Ignorar variabilidad:** modelar con un valor único lo que es aleatorio (p95/p99 cambian mucho).

Para esto es recomendable crear un checklist de variables con **tipo, unidad, fuente, temporalidad** y breve justificación.

---

## Plantilla rápida para la selacción de variables 


### Entradas (u)
- Nombre | Tipo (externa/controlable) | Unidad | Cómo varía en el tiempo | Fuente

### Estado (x)
- Nombre | Tipo (discreta/continua) | Unidad | Cota/capacidad | Por qué es necesaria

### Salidas (y)
- Métrica | Fórmula/definición | Ventana temporal | ¿Promedio/percentil/IC? | Uso en la decisión



---

## Ejercicio 2

1. Retomen el **sistema** que definieron anteriormente.
2. Llenen la **plantilla** anterior con al menos:

   * 2 **entradas** (una externa y una controlable),
   * 2 **variables de estado** (con unidades y cotas),
   * 3 **salidas** (incluyendo al menos un **percentil**).
3. Marquen **qué datos faltan** y cómo los obtendrían.

Las **variables** son el **vocabulario operativo** de tu modelo. Si las nombras, mides y relacionas bien, tu simulación será **útil** para la toma de decisiones.
